---
title: "State and Graph Control"
draft: true
categories: [agents, workflows, langgraph, ai-engineering, search, retrieval]
---

LangGraph starts with state, nodes, and transitions. The design work happens before a model call: repository facts and investigation results belong in state, injected tools belong in runtime context, and every route must correspond to a named contract.


## A minimal Change Planner graph

The backing project owns the cumulative graph. Inspecting its topology makes the control surface concrete before the course adds retrieval branches, tests, review, and memory.


In [1]:
from IPython.display import Markdown, display
from change_planner.workflow import build_change_planner_graph

graph = build_change_planner_graph(with_review=False)
display(Markdown("```{mermaid}\n" + graph.get_graph().draw_mermaid() + "\n```"))
print("nodes:", sorted(graph.nodes))
assert {"intake", "plan", "join", "analyze", "draft", "verify", "export"}.issubset(graph.nodes)


```{mermaid}
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__(<p>__start__</p>)
	intake(intake)
	plan(plan)
	investigate_branch(investigate_branch)
	investigate_sequential(investigate_sequential)
	join(join)
	analyze(analyze)
	refine(refine)
	verify_tests(verify_tests)
	draft(draft)
	review(review)
	revise(revise)
	verify(verify)
	remember(remember)
	export(export)
	__end__(<p>__end__</p>)
	__start__ --> intake;
	analyze -.-> __end__;
	analyze -.-> draft;
	analyze -.-> refine;
	analyze -.-> verify_tests;
	draft --> review;
	intake -.-> __end__;
	intake -.-> plan;
	investigate_branch --> join;
	join -.-> __end__;
	join -.-> analyze;
	plan -.-> investigate_branch;
	refine --> plan;
	remember --> export;
	review -.-> refine;
	review -.-> revise;
	review -.-> verify;
	revise -.-> __end__;
	revise -.-> review;
	verify -.-> __end__;
	verify -.-> export;
	verify -.-> remember;
	verify_tests -.-> __end__;
	verify_tests -.-> draft;
	export --> __end__;
	review -.-> review;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

nodes: ['__start__', 'analyze', 'draft', 'export', 'intake', 'investigate_branch', 'investigate_sequential', 'join', 'plan', 'refine', 'remember', 'review', 'revise', 'verify', 'verify_tests']


The graph includes a dynamic investigation fan-out, a join, an analysis route, optional targeted verification, and a final export. The important property is not the number of nodes. It is that state makes evidence, budgets, review, and terminal reasons inspectable at each transition.

## State exposes hidden failures

Run the same fixture with the complete workflow and with graph features removed. A fluent plan can survive an ablation while the contract it was supposed to enforce disappears.


In [2]:
from change_planner.workflow import run_fixture, stream_fixture


full = run_fixture("dry-run-01")
ablations = {
    variant: run_fixture("dry-run-01", variant=variant)
    for variant in ("no_review", "no_checkpointing", "no_memory")
}
events = list(stream_fixture("dry-run-01"))
print({
    "full": (full["status"], full["artifact"]["status"]),
    "ablations": {name: (state["status"], bool(state.get("review"))) for name, state in ablations.items()},
    "stream_nodes": [event["node"] for event in events],
})
assert full["status"] == "complete"
assert ablations["no_review"]["review"]["action"] == "approve"
assert {"intake", "plan", "join", "export"}.issubset({event["node"] for event in events})
assert all(set(event) == {"node", "events"} for event in events)


{'full': ('complete', 'exported'), 'ablations': {'no_review': ('complete', True), 'no_checkpointing': ('complete', True), 'no_memory': ('complete', True)}, 'stream_nodes': ['intake', 'plan', 'investigate_branch', 'investigate_branch', 'investigate_branch', 'investigate_branch', 'join', 'analyze', 'verify_tests', 'draft', 'review', 'verify', 'remember', 'export']}


Typed state does not make an impact claim true. It makes the evidence, missing branches, verification status, and completion decision observable. Chapter 03 strengthens the prototype by making retrieval outputs versioned and recoverable.
